# Air Quality Prediction and Pollution Episode Detection

This notebook follows our 16-step flow, from defining the problem to getting a URL for the app.
Run the cells from top to bottom.

**Before you start (Kaggle):** add an air quality dataset with *+ Add Input*. The UCI Air Quality data
(`AirQualityUCI.csv` or `.xlsx`) or a UCI Beijing station file both work. A GPU is optional.

## Step 1 · Problem definition

We want to predict a pollutant one hour ahead from its past readings and the weather. Then we look at the hours where
the real value was much further from our prediction than usual, and decide whether each one looks like real unusual
pollution or just a problem with the data.

Objectives:
- predict the next hour better than simply repeating the last hour's value
- test on a later time period than the one we trained on
- flag the big misses and explain each one

Change `POLLUTANT` below to try another one (for example `NO2`, `NOx`, `C6H6`, or `PM2.5` for the Beijing data).

In [ ]:
POLLUTANT = "CO"

import glob, io, json, os, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 3.5)

## Step 2 · Data collection

The notebook looks for a CSV, Excel or ZIP file under `/kaggle/input` (or a `data` folder when run on a PC).
If it finds nothing and Internet is on, it downloads the UCI Air Quality data.

In [ ]:
def find_data_file():
    for pattern in ["/kaggle/input/**/*.csv", "/kaggle/input/**/*.xlsx", "/kaggle/input/**/*.zip", "data/*.csv"]:
        files = sorted(glob.glob(pattern, recursive=True))
        if files:
            return files[0]
    import urllib.request
    os.makedirs("data", exist_ok=True)
    try:
        urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/360/air+quality.zip", "data/air-quality.zip")
    except OSError:
        raise SystemExit("No dataset found and no Internet. Add a dataset with '+ Add Input' and run again.")
    return "data/air-quality.zip"


def read_file(path):
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    if path.lower().endswith(".zip"):
        z = zipfile.ZipFile(path)
        path = io.BytesIO(z.read(next(n for n in z.namelist() if n.lower().endswith(".csv"))))
    for options in ({"sep": ";", "decimal": ","}, {}):   # UCI uses ';' and ',' decimals; most files use ','
        if hasattr(path, "seek"):
            path.seek(0)
        table = pd.read_csv(path, **options)
        if table.shape[1] > 3:
            return table
    raise ValueError("Could not read the file")


DATA_FILE = find_data_file()
raw = read_file(DATA_FILE)
print("File:", DATA_FILE)
print("Shape:", raw.shape)
raw.head()

## Step 3 · Data preprocessing

- build one timestamp column from whatever the file has (`Date` + `Time`, `year/month/day/hour`, or `datetime`)
- turn `-200` (the UCI code for "missing") into empty values
- keep one row per hour, so gaps in the data show up as empty rows
- if the file has several stations, keep the biggest one

In [ ]:
raw.columns = [str(c).strip() for c in raw.columns]
raw = raw.loc[:, ~raw.columns.str.startswith("Unnamed")]

if {"Date", "Time"} <= set(raw.columns):
    raw = raw.dropna(subset=["Date", "Time"])
    day = pd.to_datetime(raw["Date"], format="%d/%m/%Y", errors="coerce")
    if day.isna().all():
        day = pd.to_datetime(raw["Date"], errors="coerce")
    time = pd.to_datetime(day.dt.strftime("%Y-%m-%d") + " " + raw["Time"].astype(str).str.replace(".", ":", regex=False))
elif {"year", "month", "day", "hour"} <= set(raw.columns):
    time = pd.to_datetime(raw[["year", "month", "day", "hour"]])
else:
    time_col = next(c for c in raw.columns if c.lower() in ("datetime", "date_time", "timestamp", "date"))
    time = pd.to_datetime(raw[time_col])

if "station" in raw.columns and raw["station"].nunique() > 1:
    keep = raw["station"].value_counts().index[0]
    mask = raw["station"] == keep
    raw, time = raw[mask], time[mask]
    print("Using station", keep)

df = raw.apply(pd.to_numeric, errors="coerce").replace(-200, np.nan)
df.index = pd.DatetimeIndex(time, name="time")
df = df[df.index.notna()]
df = df[~df.index.duplicated()].sort_index().dropna(axis=1, how="all").asfreq("h")

print(f"{len(df)} hours, from {df.index.min()} to {df.index.max()}")
(df.isna().mean() * 100).round(1).sort_values(ascending=False).to_frame("missing %")

## Step 4 · Exploratory data analysis

A quick look at the chosen pollutant: how it changes over the year, what a typical day looks like, and how it
relates to the other columns.

In [ ]:
ALIASES = {"CO": ["CO(GT)", "CO"], "NO2": ["NO2(GT)", "NO2"], "NOx": ["NOx(GT)", "NOx"], "C6H6": ["C6H6(GT)", "C6H6"],
           "PM2.5": ["PM2.5"], "PM10": ["PM10"], "SO2": ["SO2"], "O3": ["O3"]}
TARGET = next(c for c in ALIASES[POLLUTANT] if c in df.columns)
UNITS = {"CO(GT)": "mg/m³", "NOx(GT)": "ppb"}
UNIT = UNITS.get(TARGET, "µg/m³")
y = df[TARGET]
print(f"{POLLUTANT} is in column '{TARGET}': {y.notna().mean() * 100:.1f}% of hours have a value, average {y.mean():.2f} {UNIT}")

fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
y.resample("D").mean().plot(ax=ax[0], title=f"Daily average {POLLUTANT}")
y.groupby(df.index.hour).mean().plot(ax=ax[1], marker="o", title=f"Average {POLLUTANT} by hour of day")
ax[1].set_xlabel("hour")
plt.tight_layout()
plt.show()

df.corr()[TARGET].drop(TARGET).sort_values().to_frame("correlation with " + POLLUTANT)

## Step 5 · Feature engineering

To predict hour *t* we only use what is known before *t*:
- the pollutant 1, 2, 3 and 24 hours earlier (lag features)
- the previous hour's weather (whichever weather columns the file has)
- hour of day and weekday

Then we split by time and never shuffle: first 60% to train, next 20% to choose the model, last 20% to test.

In [ ]:
WEATHER = {"temperature": ["T", "TEMP"], "relative_humidity": ["RH"], "absolute_humidity": ["AH"],
           "pressure": ["PRES"], "dew_point": ["DEWP"], "rain": ["RAIN"], "wind_speed": ["WSPM"]}
weather = {name: next(c for c in cols if c in df.columns) for name, cols in WEATHER.items() if any(c in df.columns for c in cols)}

X = pd.DataFrame(index=df.index)
X["hour"] = df.index.hour
X["weekday"] = df.index.day_of_week
for k in (1, 2, 3, 24):
    X[f"lag_{k}"] = y.shift(k)
for name, col in weather.items():
    X[name] = df[col].shift(1)
FEATURES = list(X.columns)

# The models learn the change from the last hour; the prediction is last hour + that change.
change = y - X["lag_1"]

n = len(df)
train_end, val_end = int(n * 0.6), int(n * 0.8)
rows = np.arange(24, n)
usable = (y.notna() & X["lag_1"].notna()).to_numpy()[rows]
train = rows[(rows < train_end) & usable]
val = rows[(rows >= train_end) & (rows < val_end) & usable]
test = rows[(rows >= val_end) & usable]

print("Features:", FEATURES)
print(f"Train:      {df.index[24]:%d %b %Y} to {df.index[train_end - 1]:%d %b %Y}  ({len(train)} hours)")
print(f"Validation: {df.index[train_end]:%d %b %Y} to {df.index[val_end - 1]:%d %b %Y}  ({len(val)} hours)")
print(f"Test:       {df.index[val_end]:%d %b %Y} to {df.index[-1]:%d %b %Y}  ({len(test)} hours)")
X.iloc[24:29]

## Step 6 · Model selection

We compare three common regression models. Each one sits inside a small pipeline that first fills missing inputs
with the training median and scales them. The simple guess to beat is "persistence": next hour = last hour.

In [ ]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

try:
    XGBRegressor(n_estimators=2, device="cuda").fit(np.random.rand(20, 2), np.random.rand(20))
    DEVICE = "cuda"
except Exception:
    DEVICE = "cpu"
print("XGBoost will train on:", DEVICE)


def make_pipeline(model):
    return Pipeline([("fill_missing", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("model", model)])


def predict_level(pipe, rows_X):
    return rows_X["lag_1"].to_numpy() + pipe.predict(rows_X)


def mae(idx, prediction):
    return mean_absolute_error(y.iloc[idx], prediction)


models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=150, max_depth=10, min_samples_leaf=5, n_jobs=-1, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.05, device=DEVICE, random_state=42),
}

## Step 7 · Model training

Each model is trained on the training period only.

In [ ]:
trained = {}
for name, model in models.items():
    trained[name] = make_pipeline(model).fit(X.iloc[train], change.iloc[train])
    print("trained", name)

## Step 8 · Model evaluation

We check every model on the validation period (data it has not seen) and compare it with persistence.

In [ ]:
evaluation = pd.DataFrame({
    "validation MAE": {name: mae(val, predict_level(pipe, X.iloc[val])) for name, pipe in trained.items()}
})
evaluation.loc["Persistence (repeat last hour)", "validation MAE"] = mae(val, X["lag_1"].iloc[val])
evaluation.sort_values("validation MAE").round(3)

## Step 9 · Hyperparameter tuning

We try a few settings for Random Forest and XGBoost, still judging only on the validation period. The best model
overall is then retrained on train + validation and checked once on the test period.

In [ ]:
options = {
    "Linear Regression": [LinearRegression()],
    "Random Forest": [RandomForestRegressor(n_estimators=150, max_depth=d, min_samples_leaf=5, n_jobs=-1, random_state=42)
                      for d in (10, None)],
    "XGBoost": [XGBRegressor(n_estimators=400, max_depth=d, learning_rate=0.05, device=DEVICE, random_state=42)
                for d in (4, 6)],
}
tuning = []
for name, settings in options.items():
    for model in settings:
        pipe = make_pipeline(model).fit(X.iloc[train], change.iloc[train])
        params = {k: v for k, v in model.get_params().items() if k in ("max_depth", "n_estimators", "learning_rate")}
        tuning.append({"model": name, "settings": str(params), "validation MAE": mae(val, predict_level(pipe, X.iloc[val])),
                       "estimator": model, "pipe": pipe})
tuning = pd.DataFrame(tuning).sort_values("validation MAE")
best = tuning.iloc[0]
print(f"Best on validation: {best['model']} {best['settings']}")
tuning[["model", "settings", "validation MAE"]].round(3)

In [ ]:
# Retrain the winner on train + validation, then test once
train_val = np.concatenate([train, val])
final = make_pipeline(clone(best["estimator"])).fit(X.iloc[train_val], change.iloc[train_val])   # fresh copy
prediction = predict_level(final, X.iloc[test])
model_mae, persistence_mae = mae(test, prediction), mae(test, X["lag_1"].iloc[test])
print(f"Test MAE {model_mae:.3f} {UNIT} vs persistence {persistence_mae:.3f} -> "
      f"{(1 - model_mae / persistence_mae) * 100:.1f}% better, R² {r2_score(y.iloc[test], prediction):.3f}")

# Flag the test hours where the miss was bigger than 95% of the validation misses
val_miss = np.abs(y.iloc[val].to_numpy() - predict_level(best["pipe"], X.iloc[val]))
THRESHOLD = float(np.quantile(val_miss, 0.95))
result = pd.DataFrame({"actual": y.iloc[test], "predicted": prediction})
result["difference"] = result["actual"] - result["predicted"]
result["flagged"] = result["difference"].abs() > THRESHOLD
print(f"Flag limit: {THRESHOLD:.2f} {UNIT}. Flagged hours: {result['flagged'].sum()} of {len(result)}")

week = result.iloc[: 24 * 14]
plt.plot(week.index, week["actual"], lw=1, label="actual")
plt.plot(week.index, week["predicted"], lw=1, ls="--", label="predicted")
plt.scatter(week.index[week["flagged"]], week["actual"][week["flagged"]], color="red", s=20, zorder=3, label="flagged")
plt.title(f"{POLLUTANT}: first two weeks of the test period")
plt.ylabel(UNIT)
plt.legend()
plt.show()

### Analyzing the flagged episodes

Flagged hours less than 3 hours apart form one episode. If any input was missing, or the data had a gap in the
6 hours before, we call it a **data limitation**. If everything was there, we call it **unusual pollution**.

In [ ]:
READABLE = {"lag_1": "the reading 1 hour before", "lag_2": "the reading 2 hours before", "lag_3": "the reading 3 hours before",
            "lag_24": "the reading the day before", "temperature": "temperature", "relative_humidity": "relative humidity",
            "absolute_humidity": "absolute humidity", "pressure": "air pressure", "dew_point": "dew point",
            "rain": "rain", "wind_speed": "wind speed"}
missing_inputs = X.iloc[test].isna()
gap_before = y.isna().rolling(6, min_periods=1).sum().shift(1).reindex(result.index).fillna(0)

episodes, group = [], []
for t in list(result.index[result["flagged"]]) + [None]:
    if group and (t is None or t - group[-1] > pd.Timedelta(hours=3)):
        hits = result.loc[group]
        peak = hits["difference"].abs().idxmax()
        actual, expected = result.at[peak, "actual"], result.at[peak, "predicted"]
        direction = "higher" if actual > expected else "lower"
        missing = sorted({READABLE[c] for c in FEATURES if c in READABLE and missing_inputs.loc[group, c].any()})
        gap = int(gap_before.loc[group].max())
        when = f"{group[0]:%a} {group[0].day} {group[0]:%b %Y}, {group[0]:%H:%M}" + ("" if len(group) == 1 else f" to {group[-1]:%H:%M}")
        text = f"{when}: {POLLUTANT} was {actual:.2f} {UNIT} but the model expected about {expected:.2f}, so it was {abs(actual - expected):.2f} {direction}."
        if missing or gap:
            kind = "Data limitation"
            problems = (["missing: " + ", ".join(missing)] if missing else []) + ([f"a {gap}-hour gap just before"] if gap else [])
            text += " The model was partly guessing (" + "; ".join(problems) + "), so this looks like a data problem."
        else:
            kind = "Unusual pollution"
            text += " All inputs were there, so this looks like a real unusual change in pollution."
        episodes.append({"id": len(episodes) + 1, "when": when, "type": kind, "peak_value": round(float(actual), 3),
                         "expected_value": round(float(expected), 3), "explanation": text})
        group = []
    if t is not None:
        group.append(t)

episodes = pd.DataFrame(episodes)
print(episodes["type"].value_counts().to_string())
for line in episodes["explanation"].head(5):
    print("-", line)

## Step 10 · Model saving

We save the trained pipeline with `joblib`, plus a JSON report with the results and every episode explained.
Both files show up in Kaggle's *Output* panel.

In [ ]:
import joblib

MODEL_FILE = f"model_{POLLUTANT.replace('.', '')}.pkl"
if best["model"] == "XGBoost":
    final.named_steps["model"].set_params(device="cpu")    # so it can predict on any machine
joblib.dump(final, MODEL_FILE, compress=3)
assert np.allclose(predict_level(joblib.load(MODEL_FILE), X.iloc[test[:10]]), prediction[:10])

report = {
    "pollutant": POLLUTANT, "unit": UNIT, "data_file": os.path.basename(DATA_FILE),
    "chosen_model": best["model"], "settings": best["settings"],
    "test_period": f"{df.index[val_end]:%d %b %Y} to {df.index[-1]:%d %b %Y}",
    "test_mae": round(model_mae, 3), "persistence_mae": round(persistence_mae, 3),
    "better_than_persistence_percent": round((1 - model_mae / persistence_mae) * 100, 1),
    "flag_limit": round(THRESHOLD, 3), "features": FEATURES,
    "episode_counts": episodes["type"].value_counts().to_dict(),
    "episodes": episodes.to_dict(orient="records"),
}
with open(f"report_{POLLUTANT.replace('.', '')}.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print("Saved", MODEL_FILE, "and", f"report_{POLLUTANT.replace('.', '')}.json")

## Step 11 · Backend development (FastAPI)

The API has the three endpoints from our ML deployment notes: `GET /`, `GET /health` and `POST /predict`.
Pydantic checks the input (for example, the hour has to be between 0 and 23). The cell writes `main_api.py`.

In [ ]:
api_code = """
import joblib
import pandas as pd
from typing import Optional
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field

FEATURES = %s
model = joblib.load("%s")
app = FastAPI(title="Air Quality Prediction API")


class Reading(BaseModel):
    hour: int = Field(..., ge=0, le=23)
    weekday: int = Field(..., ge=0, le=6)
    lag_1: float = Field(..., ge=0, description="value 1 hour before")
    lag_2: Optional[float] = Field(None, ge=0)
    lag_3: Optional[float] = Field(None, ge=0)
    lag_24: Optional[float] = Field(None, ge=0)
    temperature: Optional[float] = None
    relative_humidity: Optional[float] = Field(None, ge=0, le=100)
    absolute_humidity: Optional[float] = None
    pressure: Optional[float] = None
    dew_point: Optional[float] = None
    rain: Optional[float] = Field(None, ge=0)
    wind_speed: Optional[float] = Field(None, ge=0)


@app.get("/")
def root():
    return {"message": "Air Quality Prediction API is running", "pollutant": "%s"}


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(reading: Reading):
    row = pd.DataFrame([{f: getattr(reading, f) for f in FEATURES}]).astype(float)
    value = float(row["lag_1"].iloc[0] + model.predict(row)[0])
    return {"pollutant": "%s", "unit": "%s", "prediction": round(max(0.0, value), 3)}


@app.get("/page", response_class=HTMLResponse)
def page():
    return open("index.html", encoding="utf-8").read()
""" % (FEATURES, MODEL_FILE, POLLUTANT, POLLUTANT, UNIT)

with open("main_api.py", "w", encoding="utf-8") as f:
    f.write(api_code)
print("Wrote main_api.py")

## Step 12 · Frontend development

A small web page with a form. It sends the readings to `POST /predict` and shows the answer.
(The full project also has a bigger dashboard; this is the simple version for the notebook.)

In [ ]:
fields = "".join(f'<label>{f} <input name="{f}" type="number" step="any"></label><br>' for f in FEATURES)
html = f"""<!doctype html>
<html><head><meta charset="utf-8"><title>Air quality prediction</title></head>
<body style="font-family: sans-serif; max-width: 600px; margin: 30px auto">
  <h1>Next-hour {POLLUTANT} prediction</h1>
  <form id="form">{fields}<button>Predict</button></form>
  <p id="answer"></p>
  <script>
    document.getElementById("form").onsubmit = async (e) => {{
      e.preventDefault();
      const body = {{}};
      for (const input of e.target.elements) if (input.name) body[input.name] = input.value === "" ? null : Number(input.value);
      const res = await fetch("/predict", {{method: "POST", headers: {{"Content-Type": "application/json"}}, body: JSON.stringify(body)}});
      const data = await res.json();
      document.getElementById("answer").textContent = res.ok ? `Prediction: ${{data.prediction}} ${{data.unit}}` : JSON.stringify(data.detail);
    }};
  </script>
</body></html>"""
with open("index.html", "w", encoding="utf-8") as f:
    f.write(html)
print("Wrote index.html")

## Step 13 · Model integration

The page and the API are connected through `POST /predict`. Here we send a real hour from the test period to the
API, the same way the page does, and compare the answer with the real value.

In [ ]:
try:
    import fastapi, httpx  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "httpx"], check=True)

import importlib
import main_api
importlib.reload(main_api)
from fastapi.testclient import TestClient
client = TestClient(main_api.app)

hour = X.iloc[test].dropna().index[50]
body = {k: (int(v) if k in ("hour", "weekday") else float(v)) for k, v in X.loc[hour].items()}
answer = client.post("/predict", json=body).json()
print("Sent:", body)
print("API answer:", answer, "| real value:", y[hour])

## Step 14 · Testing

A few checks: the endpoints answer, bad input is rejected with status 422, and the API gives the same numbers as
the model in this notebook.

In [ ]:
assert client.get("/").status_code == 200
assert client.get("/health").json()["status"] == "ok"
assert client.post("/predict", json={**body, "hour": 30}).status_code == 422       # hour out of range
assert client.post("/predict", json={"hour": 5}).status_code == 422                  # missing fields
assert client.get("/wrong").status_code == 404

for t in X.iloc[test].dropna().index[:20]:
    one = {k: (int(v) if k in ("hour", "weekday") else float(v)) for k, v in X.loc[t].items()}
    api_value = client.post("/predict", json=one).json()["prediction"]
    notebook_value = max(0.0, float(predict_level(final, X.loc[[t]])[0]))
    assert abs(api_value - notebook_value) < 1e-3
print("All tests passed")

## Step 15 · Deployment

We start the API with Uvicorn, the same command we use on a PC: `python -m uvicorn main_api:app`.
On Kaggle it runs inside the notebook's machine, which is enough to show it working.

In [ ]:
import subprocess, sys, time, urllib.request

server = subprocess.Popen([sys.executable, "-m", "uvicorn", "main_api:app", "--port", "8000"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        print(urllib.request.urlopen("http://127.0.0.1:8000/health").read().decode())
        break
    except OSError:
        time.sleep(1)

## Step 16 · URL generation

The app is now at the local URLs below. Kaggle can't open a notebook's server to the internet, so for a public link
we run the same files on a PC: start Uvicorn, then run `cloudflared tunnel --url http://127.0.0.1:8000`, which
prints a temporary `https://...trycloudflare.com` address.

In [ ]:
print("API:          http://127.0.0.1:8000/")
print("Health check: http://127.0.0.1:8000/health")
print("Prediction:   POST http://127.0.0.1:8000/predict  (this is the URL to use in Postman)")
print("API docs:     http://127.0.0.1:8000/docs")
print("Web page:     http://127.0.0.1:8000/page")

request = urllib.request.Request("http://127.0.0.1:8000/predict", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
print("\nLive answer from the running server:", urllib.request.urlopen(request).read().decode())

In [ ]:
server.terminate()   # stop the server when you are done